# 03 — Trend Forecasting: What's Growing in the Product Ecosystem?
## TechPulse: Product Intelligence Platform

**Objetivo:** Identificar categorías dominantes y emergentes, y proyectar su evolución futura usando Prophet. Detectar la señal del boom de IA en los datos.

**Input:** `data/processed/master_dataset.parquet`  
**Output:** 
- `data/processed/category_trends.parquet` — tendencias por categoría
- `data/processed/forecasts.parquet` — proyecciones 2025–2026
- Visualizaciones en `reports/figures/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from prophet import Prophet
import ast, re, warnings

warnings.filterwarnings('ignore')

ROOT      = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
FIGURES   = ROOT / 'reports' / 'figures'

PALETTE = {
    'primary'  : '#2563EB',
    'secondary': '#7C3AED',
    'accent'   : '#059669',
    'warning'  : '#D97706',
    'danger'   : '#DC2626',
    'gray'     : '#6B7280',
}
COLORS = [PALETTE['primary'], PALETTE['secondary'], PALETTE['accent'],
          PALETTE['warning'], PALETTE['danger'], '#0891B2', '#BE185D',
          '#065F46', '#92400E', '#1E3A5F']

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor'  : '#E5E7EB', 'axes.labelcolor': '#374151',
    'axes.titlesize'  : 13, 'axes.labelsize': 11,
    'xtick.color'     : '#6B7280', 'ytick.color': '#6B7280',
    'grid.color'      : '#F3F4F6', 'grid.linestyle': '--',
    'font.family'     : 'sans-serif', 'text.color': '#111827',
})

print("✅ Librerías cargadas")

✅ Librerías cargadas


In [ ]:
df = pd.read_parquet(PROCESSED / 'master_dataset.parquet')
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['year']       = df['created_at'].dt.year.astype(int)
df['month']      = df['created_at'].dt.month.astype(int)

def extract_topics(value):
    if pd.isna(value) or str(value) in ('', 'None', 'nan'):
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(t).strip().title() for t in parsed if t]
    except:
        pass
    cleaned = re.sub(r"[\[\]'\"{}]", '', str(value))
    return [p.strip().title() for p in cleaned.split(',') if p.strip()]

df['topics_list'] = df['topics'].apply(extract_topics)
df_topics = df.explode('topics_list').dropna(subset=['topics_list'])
df_topics = df_topics[df_topics['topics_list'].str.len() > 1]

print(f"✅ Dataset listo: {len(df):,} productos — {len(df_topics):,} registros con topic")

✅ Dataset listo: 152,556 productos — 436,456 registros con topic


## 1. Selección de categorías: dominantes + emergentes

In [ ]:
# --- 5 categorías dominantes (top por volumen total) ---
dominant_cats = (df_topics['topics_list']
                 .value_counts()
                 .head(5)
                 .index.tolist())

print("📊 Categorías dominantes (top 5 por volumen):")
for i, c in enumerate(dominant_cats, 1):
    n = df_topics[df_topics['topics_list'] == c].shape[0]
    print(f"   {i}. {c} — {n:,} productos")

# --- 5 categorías emergentes (mayor crecimiento 2021→2023/2024) ---
# Comparamos volumen en años recientes vs. histórico
early  = df_topics[df_topics['year'].between(2014, 2019)]
recent = df_topics[df_topics['year'].between(2021, 2024)]

early_counts  = early['topics_list'].value_counts()
recent_counts = recent['topics_list'].value_counts()

# Solo categorías con al menos 50 productos en ambos períodos
common = set(early_counts[early_counts >= 50].index) & \
         set(recent_counts[recent_counts >= 50].index)

growth = pd.DataFrame({
    'early_count' : early_counts,
    'recent_count': recent_counts,
}).loc[list(common)].dropna()

growth['growth_ratio'] = growth['recent_count'] / growth['early_count']

# Excluir las ya seleccionadas como dominantes
growth = growth[~growth.index.isin(dominant_cats)]

# Priorizar categorías relacionadas con IA/ML
ai_keywords = ['artificial intelligence', 'machine learning', 'ai', 'gpt',
               'automation', 'chatbot', 'data science', 'deep learning', 'llm']

def is_ai_related(cat):
    return any(kw in cat.lower() for kw in ai_keywords)

growth['is_ai'] = growth.index.map(is_ai_related)

# Top emergentes: primero AI-related, luego por growth_ratio
emerging_ai    = growth[growth['is_ai']].nlargest(3, 'growth_ratio')
emerging_other = growth[~growth['is_ai']].nlargest(2, 'growth_ratio')
emerging_cats  = emerging_ai.index.tolist() + emerging_other.index.tolist()

print(f"\n🚀 Categorías emergentes (mayor crecimiento + AI-related):")
for i, c in enumerate(emerging_cats, 1):
    ratio = growth.loc[c, 'growth_ratio']
    ai    = "🤖 AI-related" if growth.loc[c, 'is_ai'] else ""
    print(f"   {i}. {c} — {ratio:.1f}x crecimiento {ai}")

ALL_CATS = dominant_cats + emerging_cats
print(f"\n✅ Total categorías seleccionadas: {len(ALL_CATS)}")
print(f"   {ALL_CATS}")

📊 Categorías dominantes (top 5 por volumen):
   1. Tech — 52,012 productos
   2. Productivity — 33,107 productos
   3. Artificial Intelligence — 24,048 productos
   4. Developer Tools — 15,800 productos
   5. Marketing — 15,168 productos

🚀 Categorías emergentes (mayor crecimiento + AI-related):
   1. Email Marketing — 1.7x crecimiento 🤖 AI-related
   2. Email — 1.6x crecimiento 🤖 AI-related
   3. Email Newsletters — 0.5x crecimiento 🤖 AI-related
   4. Money — 8.7x crecimiento 
   5. Github — 8.5x crecimiento 

✅ Total categorías seleccionadas: 10
   ['Tech', 'Productivity', 'Artificial Intelligence', 'Developer Tools', 'Marketing', 'Email Marketing', 'Email', 'Email Newsletters', 'Money', 'Github']
